# Week 5: MLOps Introduction

## Learning Objectives
- Understand MLOps principles and tooling
- Use MLflow for experiment tracking
- Create reproducible ML pipelines
- Implement CI/CD for ML models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import json
import time

sns.set_theme(style='whitegrid')

## 1. What is MLOps?

MLOps bridges **DevOps** and **Machine Learning** to:
- Automate ML lifecycle management
- Enable continuous training and deployment
- Ensure reproducibility and auditability
- Scale model serving

**Key Tools:**
| Stage | Tools |
|-------|-------|
| Experiment tracking | MLflow, Weights & Biases |
| Data versioning | DVC |
| Pipeline orchestration | Airflow, Prefect, Kubeflow |
| Model registry | MLflow, SageMaker |
| Serving | BentoML, Seldon, TorchServe |
| Monitoring | Evidently, WhyLabs |

## 2. MLflow Experiment Tracking

In [ ]:
try:
    import mlflow
    import mlflow.sklearn

    cancer = load_breast_cancer()
    X, y = cancer.data, cancer.target
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42)

    mlflow.set_experiment('breast-cancer-classification')

    for n_estimators in [50, 100, 200]:
        with mlflow.start_run():
            params = {'n_estimators': n_estimators, 'max_depth': 5, 'random_state': 42}
            model = Pipeline([('s', StandardScaler()),
                               ('m', RandomForestClassifier(**params))])
            model.fit(X_train, y_train)
            acc = accuracy_score(y_test, model.predict(X_test))
            auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])

            mlflow.log_params(params)
            mlflow.log_metric('accuracy', acc)
            mlflow.log_metric('roc_auc', auc)
            mlflow.sklearn.log_model(model, 'model')
            print(f'n_estimators={n_estimators}: acc={acc:.4f}, auc={auc:.4f}')

    print('Runs logged. Start MLflow UI with: mlflow ui')
except ImportError:
    print('MLflow not installed: pip install mlflow')

## 3. A Simple Manual Experiment Tracker

In [ ]:
class ExperimentTracker:
    def __init__(self, experiment_name):
        self.experiment_name = experiment_name
        self.runs = []

    def log_run(self, params, metrics):
        run = {
            'id': len(self.runs) + 1,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'params': params,
            'metrics': metrics
        }
        self.runs.append(run)
        print(f'Run {run["id"]}: {metrics}')
        return run

    def best_run(self, metric):
        return max(self.runs, key=lambda r: r['metrics'].get(metric, 0))

    def to_dataframe(self):
        rows = []
        for run in self.runs:
            row = {'id': run['id'], 'timestamp': run['timestamp']}
            row.update({f'param_{k}': v for k, v in run['params'].items()})
            row.update({f'metric_{k}': v for k, v in run['metrics'].items()})
            rows.append(row)
        return pd.DataFrame(rows)

tracker = ExperimentTracker('breast-cancer-model')
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

for n_est in [50, 100, 200]:
    model = RandomForestClassifier(n_estimators=n_est, random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    tracker.log_run({'n_estimators': n_est}, {'auc': auc})

print('\nBest run:', tracker.best_run('auc')['params'])
print('\nAll runs:')
print(tracker.to_dataframe().round(4))

## 4. CI/CD for ML

In [ ]:
cicd_config = '''
# .github/workflows/ml-pipeline.yml
name: ML Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v3
    - uses: actions/setup-python@v4
      with:
        python-version: 3.9

    - name: Install dependencies
      run: pip install -r requirements.txt

    - name: Run unit tests
      run: pytest tests/

    - name: Train model
      run: python src/train.py

    - name: Evaluate model
      run: python src/evaluate.py

    - name: Check performance threshold
      run: python src/check_threshold.py --min-auc 0.90
'''
print(cicd_config)

## Exercise

1. Set up MLflow locally and track 10 experiments with different hyperparameters
2. Use DVC to version a dataset and model
3. Write a GitHub Actions workflow that trains, tests, and evaluates a model

In [ ]:
# Your code here


## Congratulations! You have completed the 12-Month Data Science Bootcamp!

### What You've Learned

| Month | Topic |
|-------|-------|
| 1  | Python Programming Basics |
| 2  | Data Analysis with Pandas and NumPy |
| 3  | Data Visualization |
| 4  | Statistics and Probability |
| 5  | Data Cleaning and Preprocessing |
| 6  | Exploratory Data Analysis |
| 7  | Introduction to Machine Learning |
| 8  | Advanced Machine Learning |
| 9  | Deep Learning |
| 10 | Natural Language Processing |
| 11 | Time Series Analysis |
| 12 | End-to-End Projects and MLOps |

### Next Steps
- Build a portfolio of 3-5 end-to-end projects
- Contribute to open-source data science projects
- Earn certifications (Google ML, AWS ML Specialty, DataCamp)
- Join Kaggle competitions
- Network on LinkedIn and GitHub

**Keep learning, keep building!** 🚀